# 01_Bronze_Data_Ingestion

Run on Databricks serverless with the uploaded Olist files. Original team work: review, execute, validate and document results.

In [ ]:
CATALOG = "workspace"
SCHEMA = "ecommerce_bigdata"
VOLUME = "raw_data"
RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print('Raw data path:', RAW_PATH)

In [ ]:
from pyspark.sql import functions as F
files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}
for name, filename in files.items():
    path = f"{RAW_PATH}/{filename}"
    df = (spark.read.option("header",True).option("inferSchema",False)
          .option("multiLine",True).option("escape",'"').csv(path))
    df = df.toDF(*[col.lstrip("\ufeff") for col in df.columns])
    df = df.withColumn("_ingested_at",F.current_timestamp()).withColumn("_source_file",F.lit(filename))
    table = f"{CATALOG}.{SCHEMA}.bronze_{name}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(table)
    print(table, df.count())